# Demo 1: фокусна відстань, поле зору та зум

До слайдів **«Кут огляду (Field of View): FOV & Фокусна відстань»** і **«Прокляття масштабу»** (Optical vs Digital Zoom).

1. **FOV і фокусна відстань** — що бачить камера з конкретною матрицею та об'єктивом.
2. **Цифровий зум проти другого сенсора** — чому довший об'єктив не замінити інтерполяцією.

Обидва блоки одразу відкриваються з фото `artifacts/images/lesson_1_demo_1_rgb.jpg` (2752 px завширшки, FOV 120°). Своє можна обрати кнопкою **«Обрати фото»** — бажано від ~8000 px завширшки: на меншому сильний зум домальовує деталі інтерполяцією, і панель про це попередить.

> Формули — у клітинках нижче, віджети — у `panels.py`. Змінив формулу → перезапусти її клітинку **і клітинку з панеллю**: панель бере функції в момент запуску. Змінив `panels.py` → перезапусти ядро.

## Блок 1: фокусна відстань і поле зору

1. **Фото** — одразу відкривається `artifacts/images/lesson_1_demo_1_rgb.jpg`. Своє можна підставити кнопкою **«Обрати фото»** — JPG або PNG (HEIC з iPhone OpenCV не читає).
2. **FOV оригіналу** — горизонтальний кут, під яким знято фото. Ширшого кута симуляція не дасть. Орієнтири: смартфон 65–75°, DJI ~84°, GoPro wide ~120°, 50 мм на повному кадрі ~40°.
3. **Формат матриці** — зі списку або вручну, у мм.
4. Рухай повзунок **f**.

**Зліва** — оригінал із рамкою кадру камери та кутом огляду. **Справа** — кадр камери, кратність зуму й кількість **реальних** пікселів із фото: що довша фокусна, то менше справжньої деталі. Це тема блоку 2.

**Модель** — прямолінійна оптика: `tan(FOV/2) = sensor / (2f)`. Ширина рамки — частка `tan(FOV/2) / tan(FOV_оригіналу/2)` від ширини фото, висота — за пропорціями матриці. Якщо матриця «вища» за фото, рамка впирається в його краї — панель попередить. Для fisheye (`r ∝ θ`) модель не працює.

In [3]:
import numpy as np

MIN_FOV_DEG = 10.0


def half_tan(fov_deg: float) -> float:
    """tan(FOV/2): у прямолінійній оптиці саме ця величина пропорційна розміру матриці."""
    return np.tan(np.radians(fov_deg / 2))


def fov_from_focal(sensor_mm: float, focal_mm: float) -> float:
    """Кут огляду (°) уздовж сторони матриці розміром sensor_mm."""
    return 2 * np.degrees(np.arctan(sensor_mm / (2 * focal_mm)))


def focal_from_fov(sensor_mm: float, fov_deg: float) -> float:
    """Обернена до fov_from_focal: яка фокусна дає заданий кут."""
    return sensor_mm / (2 * half_tan(fov_deg))


def zoom_factor(orig_fov_deg: float, fov_deg: float) -> float:
    """У скільки разів лінійно більшим стає об'єкт порівняно з оригіналом."""
    return half_tan(orig_fov_deg) / half_tan(fov_deg)


def crop_box(img_w: float, img_h: float, orig_fov_deg: float, fov_deg: float,
             sensor_aspect: float) -> tuple[float, float, float, float, bool]:
    """Рамка (x0, y0, w, h) у пікселях оригіналу для кута fov_deg + чи обрізано по вертикалі.

    Ширина — частка ПОВНОЇ ширини фото, бо саме їй відповідає orig_fov_deg. Висота — з аспекту
    матриці; якщо рамка вища за фото, звужуємо її зі збереженням аспекту й піднімаємо clipped.
    """
    w = img_w * min(1.0, 1 / zoom_factor(orig_fov_deg, fov_deg))
    h = w / sensor_aspect
    clipped = h > img_h
    if clipped:
        w, h = img_h * sensor_aspect, img_h
    return (img_w - w) / 2, (img_h - h) / 2, w, h, clipped

In [4]:
from panels import launch_fov

launch_fov(fov_from_focal, focal_from_fov, zoom_factor, crop_box, MIN_FOV_DEG)

## Блок 2: цифровий зум проти другого сенсора

Дві камери з однаковою матрицею: одна лише з wide-об'єктивом, друга — ще й з теле. Уся різниця — в оптиці.

1. Фото й **FOV оригіналу** — як у блоці 1.
2. **Роздільність сенсорів** — фото стискається до неї, і далі камери бачать лише ці пікселі.
3. **FOV теле-лінзи** — вужчий за FOV оригіналу.
4. Рухай повзунок **Зум ×**.

- **Згори** — wide-кадр: пунктир — поле теле-лінзи, суцільна рамка — поточний зум.
- **Знизу ліворуч** — лише цифровий зум: кроп розтягується до роздільності матриці, нових деталей немає.
- **Знизу праворуч** — wide + теле. Поки зум не дійшов до кута теле-лінзи, кадри однакові. Далі камера перемикається на теле-сенсор, який знімає ту саму ділянку власними пікселями, а потім знову зумить цифрово — вже поверх теле, як гібридний зум у смартфонах.

Точка перемикання — у скільки разів теле-кроп вужчий за wide-кроп. Зазвичай це `zoom_factor(FOV_оригіналу, FOV_теле)`; якщо матриця «вища» за фото (4:3 на широкому фото), wide-кроп упирається у верх і низ і звужується, тож перемикання настає раніше.

> Симуляція чесна, поки у фото більше пікселів, ніж у теле-сенсора на його куті. Інакше панель попередить.

In [5]:
import cv2


def sensor_capture(photo: np.ndarray, photo_fov_deg: float, lens_fov_deg: float,
                   out_res: tuple[int, int]) -> np.ndarray:
    """Кадр сенсора out_res через лінзу lens_fov: кутовий кроп з оригіналу + ресайз під матрицю."""
    img_h, img_w = photo.shape[:2]
    x0, y0, box_w, box_h, _ = crop_box(img_w, img_h, photo_fov_deg, lens_fov_deg,
                                       out_res[0] / out_res[1])
    crop = photo[int(y0):int(y0 + box_h), int(x0):int(x0 + box_w)]
    return cv2.resize(crop, out_res, interpolation=cv2.INTER_AREA)


def digital_zoom(frame: np.ndarray, zoom: float, out_res: tuple[int, int]) -> np.ndarray:
    """Цифровий зум: центральний кроп уже знятого кадру, розтягнутий назад до роздільності матриці.

    Нових деталей не додає — лише інтерполює наявні пікселі.
    """
    h, w = frame.shape[:2]
    crop_w, crop_h = max(1, int(w / zoom)), max(1, int(h / zoom))
    x0, y0 = (w - crop_w) // 2, (h - crop_h) // 2
    crop = frame[y0:y0 + crop_h, x0:x0 + crop_w]
    return cv2.resize(crop, out_res, interpolation=cv2.INTER_CUBIC)


def hybrid_frame(wide: np.ndarray, tele: np.ndarray, zoom: float, switch_at: float,
                 out_res: tuple[int, int]) -> tuple[np.ndarray, bool]:
    """Права камера: до switch_at — цифровий зум wide-сенсора, далі — теле-сенсор із залишковим зумом.

    Матриця в обох камер однакова, тож різниця між половинами — виключно в об'єктиві.
    """
    if zoom < switch_at:
        return digital_zoom(wide, zoom, out_res), False
    return digital_zoom(tele, zoom / switch_at, out_res), True

In [ ]:
from panels import launch_zoom

launch_zoom(sensor_capture, digital_zoom, hybrid_frame, crop_box)